# Assignment 9: Hypothesis Testing (Part 1)

## Objective

In many situations, we cannot get the full population but only a sample. If we derive an interesting result from a sample, how likely can we derive the same result from the entire population? In other words, we want to know whether this result is a true finding or it just happens in the sample by chance. Hypothesis testing aims to answer this fundamental question. 


**Hypothesis Testing**
1. Why A/B testing?  
2. What is a permutation test? How to implement it?
3. What is p-value? How to avoid p-hacking? 
4. What is a chi-squared test? How to implement it?


## Task 1. A/B Testing
> Acknowledgment: Thank [Greg Baker](http://www.cs.sfu.ca/~ggbaker/) for helping me to prepare this task.

A very common technique to evaluate changes in a user interface is A/B testing: show some users interface A, some interface B, and then look to see if one performs better than the other.

Suppose I started an A/B test on CourSys. Here are the two interfaces that I want to compare with. I want to know whether a good placeholder in the search box can attract more users to use the `search` feature.


![](img/ab-testing.png)

The provided [searchlog.json](searchlog.json) has information about users' usage. The question I was interested in: is the number of searches per user different?

To answer this question, we need to first pick up a **test statistic** to quantify how good an interface is. Here, we choose "the search_count mean". 

Please write the code to compute **the difference of the search_count means between interface A and Interface B.** 

In [2]:
import json

a_counts = []
b_counts = []

with open('searchlog.json') as f:
    for line in f:
        row = json.loads(line)
        if row['search_ui'] == 'A':
            a_counts.append(row['search_count'])
        elif row['search_ui'] == 'B':
            b_counts.append(row['search_count'])

mean_a = sum(a_counts) / len(a_counts)
mean_b = sum(b_counts) / len(b_counts)
mean_diff = mean_a - mean_b

print('Mean of interface A:', mean_a)
print('Mean of interface B:', mean_b)
print('Difference in means (A - B):', mean_diff)

Mean of interface A: 0.6637931034482759
Mean of interface B: 0.7987987987987988
Difference in means (A - B): -0.13500569535052287


Suppose we find that the mean value increased by 0.135. Then, we wonder whether this result is just caused by random variation. 

We define the Null Hypothesis as
 * The difference in search_count mean between Interface A and Interface B is caused by random variation. 
 
Then the next job is to check whether we can reject the null hypothesis or not. If it does, we can adopt the alternative explanation:
 * The difference in search_count mean  between Interface A and Interface B is caused by the design differences between the two.

We compute the p-value of the observed result. If p-value is low (e.g., <0.01), we can reject the null hypothesis, and adopt  the alternative explanation.  

Please implement a permutation test (numSamples = 10000) to compute the p-value. Note that you are NOT allowed to use an implementation in an existing library. You have to implement it by yourself.

In [3]:
import random

numSamples = 10000
all_counts = a_counts + b_counts
size_a = len(a_counts)
observed_diff = abs(mean_diff)
count = 0

for _ in range(numSamples):
    shuffled = all_counts[:]
    random.shuffle(shuffled)

    sample_a = shuffled[:size_a]
    sample_b = shuffled[size_a:]

    sample_mean_a = sum(sample_a) / len(sample_a)
    sample_mean_b = sum(sample_b) / len(sample_b)
    sample_diff = abs(sample_mean_a - sample_mean_b)

    if sample_diff >= observed_diff:
        count += 1

p_value = count / numSamples
print('p-value:', p_value)

p-value: 0.2585


Suppose we want to use the same dataset to do another A/B testing. We suspect that instructors are the ones who can get more useful information from the search feature, so perhaps non-instructors didn't touch the search feature because it was genuinely not relevant to them.

So we decide to repeat the above analysis looking only at instructors.

**Q. If using the same dataset to do this analysis, do you feel like we're p-hacking? If so, what can we do with it?**

**A.** Yes, this can be considered p-hacking if we decided to look only at instructors after seeing the result from the full dataset. In that case, we are changing the hypothesis after looking at the data and searching for a subgroup that gives a more interesting result. This increases the chance of finding a false positive by random luck. A better approach is to decide this subgroup analysis before examining the data, report it clearly as an exploratory analysis, apply a multiple-testing correction if many subgroup analyses are tried, and ideally confirm the result with a new independent dataset.

## Task 2. Chi-squared Test 

There are tens of different hypothesis testing methods. It's impossible to cover all of them in one week. Given that this is an important topic in statistics, I highly recommend using your free time to learn some other popular ones such as <a href="https://en.wikipedia.org/wiki/Chi-squared_test">Chi-squared test</a>, <a href="https://en.wikipedia.org/wiki/G-test">G-test</a>, <a href="https://en.wikipedia.org/wiki/Student%27s_t-test">T-test</a>, and <a href="https://en.wikipedia.org/wiki/Mann%E2%80%93Whitney_U_test">Mann–Whitney U test</a>.

On the searchlog dataset, there are two categorical columns: `is_instructor` and `search_ui`. In this task, your job is to first learn how a Chi-Squired test works by yourself and then use it to test whether `is_instructor` and `search_ui` are correlated. 

Please write code to compute the Chi-squared stat. Note that you are **not** allowed to call an existing function (e.g., stats.chi2, chi2_contingency). 

In [4]:
import json

observed = {
    True: {'A': 0, 'B': 0},
    False: {'A': 0, 'B': 0}
}

with open('searchlog.json') as f:
    for line in f:
        row = json.loads(line)
        observed[row['is_instructor']][row['search_ui']] += 1

row_totals = {
    True: observed[True]['A'] + observed[True]['B'],
    False: observed[False]['A'] + observed[False]['B']
}
col_totals = {
    'A': observed[True]['A'] + observed[False]['A'],
    'B': observed[True]['B'] + observed[False]['B']
}
total = row_totals[True] + row_totals[False]

chi_squared = 0
for is_instructor in [True, False]:
    for search_ui in ['A', 'B']:
        expected = row_totals[is_instructor] * col_totals[search_ui] / total
        chi_squared += (observed[is_instructor][search_ui] - expected) ** 2 / expected

print('Observed counts:', observed)
print('Chi-squared stat:', chi_squared)

Observed counts: {True: {'A': 115, 'B': 120}, False: {'A': 233, 'B': 213}}
Chi-squared stat: 0.6731740891275046


Please explain how to use Chi-squared test to determine whether `is_instructor` and `search_ui` are correlated. 

**A.** To use the Chi-squared test, we first build a contingency table of the observed counts for each combination of `is_instructor` and `search_ui`. Then we compute the expected count for each cell under the null hypothesis that the two variables are independent. The Chi-squared statistic is the sum of `(observed - expected)^2 / expected` over all cells. If the statistic is large, the observed counts are far from the expected counts, which suggests that the variables may be correlated. If the statistic is small, the observed counts are close to the expected counts, which suggests little or no evidence of correlation. In our case, the Chi-squared statistic is about 0.673, which is small, so there is no strong evidence that `is_instructor` and `search_ui` are correlated.

## Submission

Complete the code in this notebook, and submit it to Canvas